In [96]:
import pandas as pd
import numpy as np
import plotly.express as px
import pickle

In [153]:
class Linearregression:
    def __init__(self, learning_rate = 0.001, convergence_tool = 1e-6):
        self.learning_rate = learning_rate
        self.convergence_tool = convergence_tool
        self.w = None
        self.b = None

    def params_init(self, n_features):
        self.w = np.random.rand(n_features) * 0.001
        self.b = 0

    def forward(self,X):
        return np.dot(X, self.w) + self.b

    def cost_compute(self, predictions):
        m = len(predictions)
        cost = np.sum((predictions - self.y) ** 2) / (2 * m)

        return cost

    def backward(self, predictions):
        m = len(predictions)
        error = predictions - self.y
        self.dw = np.dot(error , self.X) / m
        self.db = np.sum(error) / m

    def fit(self, X, y, iterations, plot_cost=True):

        assert isinstance(X, np.ndarray), "X must be an array"
        assert isinstance(y, np.ndarray), "y. must be an array"
        assert len(X) == len(y), "X and y must be the same lenght"
        assert iterations > 0, "Iterations should be greater than 0"

        self.X = X
        self.y = y 
        self.params_init(X.shape[1])
        cost_history = []

        for i in range(iterations):
            predictions = self.forward(X)
            cost = self.cost_compute(predictions)
            self.backward(predictions)
            self.w -= self.learning_rate * self.dw
            self.b -= self.learning_rate * self.db

            cost_history.append(cost)

            if i % 100 == 0:
                print(f"iteration {i}, cost: {cost}")

            if i > 0 and abs(cost_history[-1] - cost_history[-2]) < self.convergence_tool:
                print(f"converged at {i} iteration")
                break

        if plot_cost:
            fig = px.line(y=cost_history, title="cost plot",template='plotly_dark')
            fig.update_layout(
                title_font_color="#41BEE9",
                xaxis=dict(color="#41BEE9", title="Iterations"),
                yaxis=dict(color="#41BEE9", title="Cost")
            )

            fig.show()

    def predict(self, X):
        return self.forward(X)

    def save_model(self, filename=None):
        model_data = {
            'learning_rate': self.learning_rate,
            'convergence_tol': self.convergence_tol,
            'W': self.W,
            'b': self.b
        }

        with open(filename, 'wb') as file:
            pickle.dump(model_data, file)

    def evaluation_model(self, y, prediccion):
        mse = np.mean((y - prediccion) ** 2)
        rmse = np.sqrt(mse)
        ss_res = np.sum((y - prediccion) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1 - ss_res / ss_tot

        print(f"mse: {mse}")
        print(f"rmse: {rmse}")
        print(f"r2: {r2}")

In [171]:
PATH = r"/Users/samuelsantiagosuarezpaez/Documents/Personal_Projects/data/Student_Performance.csv"
df = pd.read_csv(PATH)

df['Extracurricular Activities'] = df['Extracurricular Activities'].map({'Yes':1,'No':0})

X,y = df.drop(columns=['Performance Index','Previous Scores']), df['Performance Index']
df.head(2)

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,1,9,1,91.0
1,4,82,0,4,2,65.0


In [172]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.3)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train, y_test = y_train.values, y_test.values

In [173]:
model_linear = Linearregression(learning_rate=0.001)

model_linear.fit(X_train_scaled, y_train,iterations=100000)

iteration 0, cost: 1706.9941740278134
iteration 100, cost: 1426.093174638487
iteration 200, cost: 1196.1375899691434
iteration 300, cost: 1007.8877021884961
iteration 400, cost: 853.7795623638276
iteration 500, cost: 727.6210619965983
iteration 600, cost: 624.343127257982
iteration 700, cost: 539.7960384230962
iteration 800, cost: 470.5826902388091
iteration 900, cost: 413.9220933299105
iteration 1000, cost: 367.5376318988941
iteration 1100, cost: 329.5655877339341
iteration 1200, cost: 298.4802548803013
iteration 1300, cost: 273.0326359749482
iteration 1400, cost: 252.20025697997355
iteration 1500, cost: 235.14608380719284
iteration 1600, cost: 221.18489005509105
iteration 1700, cost: 209.75572447675057
iteration 1800, cost: 200.3993718935806
iteration 1900, cost: 192.73990191329054
iteration 2000, cost: 186.4695640638759
iteration 2100, cost: 181.33642241848776
iteration 2200, cost: 177.13423286193225
iteration 2300, cost: 173.69415626125473
iteration 2400, cost: 170.8779745712849
it

In [174]:
predicciones_ = model_linear.predict(X_test_scaled)
model_linear.evaluation_model(y_test,predicciones_)

mse: 315.4978295709502
rmse: 17.762258571785015
r2: 0.16285953566251388


In [175]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

sk_model = LinearRegression()
sk_model.fit(X_train_scaled, y_train)
sk_pred = sk_model.predict(X_test_scaled)

print("sklearn R²:", r2_score(y_test, sk_pred))
print("sklearn RMSE:", np.sqrt(mean_squared_error(y_test, sk_pred)))

sklearn R²: 0.16287589842112749
sklearn RMSE: 17.762084980266


In [ ]:
import statsmodels.api as sm

X_train_scaled
X_con_constante = sm.add_constant(X_train_scaled) 

modelo = sm.OLS(y_train, X_con_constante).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.135
Model:                            OLS   Adj. R-squared:                  0.135
Method:                 Least Squares   F-statistic:                     273.2
Date:                Tue, 08 Sep 2026   Prob (F-statistic):          1.60e-218
Time:                        22:00:11   Log-Likelihood:                -30081.
No. Observations:                7000   AIC:                         6.017e+04
Df Residuals:                    6995   BIC:                         6.021e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         55.2110      0.213    259.626      0.0

# Lectura

x1 tiene un coeficiente de 6.909, lo que significa que por cada aumento de una unidad en x1, el target aumenta aproximadamente 6.909 unidades. El t-statistic es 32.484, indicando que el coeficiente está 32.484 errores estándar alejado de 0. Su p-value es extremadamente pequeño, por lo que rechazamos H₀ de que el coeficiente sea 0. Por tanto, x1 aporta información estadísticamente significativa para explicar el target. Finalmente, el intervalo de confianza del 95% para el verdadero coeficiente está entre 6.492 y 7.326.